<a href="https://colab.research.google.com/github/mjmj002/Sparta/blob/main/Python/day3/day3_LLM_huggingface_%EB%94%B0%EB%9D%BC%EC%9E%A1%EA%B8%B0_%EB%8B%B5%EC%95%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day11 실습 — HuggingFace + LLM 기초 실습

이번 실습에서는 **LLM과 HuggingFace의 기본 사용 방법**을 간단하게 실습합니다.

목표
- HuggingFace `pipeline` 사용하기
- 감정 분석 모델 사용하기
- 토크나이저 → 모델 → 결과 흐름 이해하기

이 실습은 **최소 코드로 LLM을 사용하는 경험**을 만드는 것이 목적입니다.

ai 모델들의 저장소
- pipeline :input 데이터 들어오면 정해진 수순에 따라 처리가 이뤄지고 처리된게 output으로 나옴. ex) 배관 정화시설 수도꼭지

## 1. 라이브러리 설치

In [ ]:
# 필요 라이브러리 설치
# 처음 실행 시 주석을 해제하세요

!pip install transformers torch

## 2. pipeline으로 감정 분석 실행

In [ ]:

# from 어디서? 위 셀에서 pip python install package 를 사용해서 다운로드한
# transformers 라는 lib, module (라이브러리 혹은 모듈) 을 호출해서 사용할거다.
# 어떤걸? import pipeline (트랜스포머에 있는 파이프라인을)
from transformers import pipeline

# classifier는 지금 pipeline으로 들고 온 함수중에서
# 트랜스포머 라이브러리 중에 하나의 감정 분석 모델을 들고온 것

# 감정 분석 파이프라인 생성
classifier = pipeline("sentiment-analysis") # 감정 분석
# classifier 구분자. 분류기

text = "I love studying AI."

# 감정분석기 파이프 라인 안에 스트링을 text를 넣었더니,
# 결과가 이렇다.
result = classifier(text)

print(result)

# 결과가 긍정적이고 긍정 점수가 1에 가까움
# 이유 : love 와 study 로 인해 긍정

# model.safetensors : 모델의 저장된 차원의 수 데이터 를 불러오겠다. -> 모델이 286메가
# param=pre_classifier.weight : 분류기의 weight 를 가져오겠다.
# tokenizer_config : 숫자들을 가져오는 것

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9992775321006775}]



### 확인하기

결과에는 다음 정보가 포함됩니다.

- label (POSITIVE / NEGATIVE)
- score (확률)

모델은 내부적으로 다음 과정을 거칩니다.

text → tokenizer → model → logits → softmax → label

- 텍스트 들어온 걸 토크나이저 하고 모델에서 추론 과정 거치고 분류하고 라벨을 통해 나옴

- sentiment-analysis 는 이 과정을 통해 추론됨

## 실습 문제 1 — 문장 바꿔보기

In [ ]:
# 아래 문장을 바꾸어 감정 분석 결과를 확인해보세요.

text = "This insurance service is terrible."

result = classifier(text)

print(result)
# 출력 예시: [{'label': 'NEGATIVE', 'score': 0.9997}]

# label 과 score 만 따로 출력
print(f"감정: {result[0]['label']}")
print(f"확률: {result[0]['score']:.4f}")

[{'label': 'NEGATIVE', 'score': 0.9996746778488159}]
감정: NEGATIVE
확률: 0.9997


## 실습 문제 2 — 여러 문장 분석

In [ ]:
# 아래 문장 리스트를 반복문으로 감정 분석해보세요
# 우리는 리스트를 textx 라는 변수에 담아서 만들겠습니다.
# 총 3개, 리스트의 크기, 길이는 3인 리스트를 문자열들을 넣어서 만들었습니다.

texts = [
    "I am very happy with this insurance.",
    "This service is slow and frustrating.",
    "The support team was helpful."
]

# 첫줄부터 하나씩 반복하면서, text 라는 변수에 담아서 반복을 하겠다.
# 첫 줄은 classfifier 들어가서 분석되고, 프린트가 찍힌다.
# 방법 1 — 반복문으로 하나씩 분석
for text in texts:
    result = classifier(text)
    print(f"문장: {text}")
    print(f"  → 감정: {result[0]['label']}, 확률: {result[0]['score']:.4f}")
    print()

# 방법 2 — pipeline에 리스트를 한 번에 전달 (더 효율적)
results = classifier(texts)
print(results)
for text, result in zip(texts, results): # zip : 묶어서 실행함
    print(f"{result['label']:8s} ({result['score']:.4f})  |  {text}")

문장: I am very happy with this insurance.
  → 감정: POSITIVE, 확률: 0.9999

문장: This service is slow and frustrating.
  → 감정: NEGATIVE, 확률: 0.9996

문장: The support team was helpful.
  → 감정: POSITIVE, 확률: 0.9996

[{'label': 'POSITIVE', 'score': 0.9998737573623657}, {'label': 'NEGATIVE', 'score': 0.9995900988578796}, {'label': 'POSITIVE', 'score': 0.9995877146720886}]
POSITIVE (0.9999)  |  I am very happy with this insurance.
NEGATIVE (0.9996)  |  This service is slow and frustrating.
POSITIVE (0.9996)  |  The support team was helpful.


## 3. tokenizer 직접 확인

In [ ]:
# transformers의 AutoTokenizer를 불러오겠다. 사용하겠다.
from transformers import AutoTokenizer

# AutoTokenizer는 .from_pertrained
# 사전에 학습된 토크나이저를 불러오겠다.
# 모델의 이름은 distilbert-base-uncased . 증류된 모델의 베이스를 가져오겠다
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

text = "AI is changing the world."
# 토크나이저에 이 텍스트를 넣어봤더니 [0.0.0....] attention_mask ~ 나옴

tokens = tokenizer(text)

print(tokens)
# 출력 예시:
# {'input_ids': [101, 1ai, ...], 'attention_mask': [1, 1, ...]}

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': [101, 9932, 2003, 5278, 1996, 2088, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


tokenizer는 문장을 **모델이 이해할 수 있는 숫자 형태**로 바꿉니다.

주요 값

- input_ids : 단어를 숫자 ID로 변환한 값
- attention_mask : 실제 토큰(1)과 패딩(0)을 구분하는 값

패딩 : 빈 공간 없게 둘러싼 값

## 실습 문제 3 — 토큰 확인

In [ ]:
# 아래 문장을 토큰화하고 input_ids를 출력하세요

text = "Insurance claims are processed quickly."

tokens = tokenizer(text)

# input_ids 출력
print("input_ids:", tokens['input_ids'])

# 토큰 ID를 다시 단어로 변환해서 확인 (어떻게 쪼개졌는지 확인)
decoded = tokenizer.convert_ids_to_tokens(tokens['input_ids'])
print("tokens  :", decoded)

# 토큰 수 확인
print(f"토큰 수  : {len(tokens['input_ids'])}개")

# 101 102 시작과 끝을 알리는 패딩. 0 1 로 나오지 않음
# 101~102 가 패딩들.
# convert_ids_to_tokens : 아이디를 토큰으로 바꾸겠다.
# 5427 -> insurance / 2024 -> are  이런식으로 매칭이 됨

input_ids: [101, 5427, 4447, 2024, 13995, 2855, 1012, 102]
tokens  : ['[CLS]', 'insurance', 'claims', 'are', 'processed', 'quickly', '.', '[SEP]']
토큰 수  : 8개


## 도전 과제 (고급)

### 도전 1. 한국어 문장 감정 분석

In [ ]:
# 한국어를 지원하는 모델로 교체해야 합니다.
# distilbert는 영어 전용 → 한국어 모델 사용

ko_classifier = pipeline(
    "sentiment-analysis",
    model="snunlp/KR-FinBert-SC"   # 한국어 금융 감성 분석 모델
    # 모델은 한국거로 하겠다.
)

# 모델만 다르지 위 아래 내용은 같음

ko_texts = [
    "오늘 비트코인이 많이 올랐네요.",
    "보험금 청구가 너무 오래 걸려서 답답합니다.",
    "상담원이 친절하게 안내해줬어요."
]

for text in ko_texts:
    result = ko_classifier(text)
    print(f"문장: {text}")
    print(f"  → 감정: {result[0]['label']}, 확률: {result[0]['score']:.4f}")
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: snunlp/KR-FinBert-SC
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


문장: 오늘 비트코인이 많이 올랐네요.
  → 감정: neutral, 확률: 0.9998

문장: 보험금 청구가 너무 오래 걸려서 답답합니다.
  → 감정: neutral, 확률: 0.7920

문장: 상담원이 친절하게 안내해줬어요.
  → 감정: neutral, 확률: 0.9995



### 도전 2. 여러 문장을 한 번에 분석

In [ ]:
# pipeline에 리스트를 직접 전달하면 한 번에 처리됩니다.
# 반복문보다 빠릅니다.

texts = [
    "I am very happy with this insurance.",
    "This service is slow and frustrating.",
    "The support team was helpful."
]

results = classifier(texts)   # 리스트를 한 번에 전달

for text, result in zip(texts, results):
    print(f"{result['label']:8s} ({result['score']:.4f})  |  {text}")

POSITIVE (0.9999)  |  I am very happy with this insurance.
NEGATIVE (0.9996)  |  This service is slow and frustrating.
POSITIVE (0.9996)  |  The support team was helpful.


### 도전 3. 결과를 표 형태로 출력

In [ ]:
import pandas as pd

texts = [
    "I am very happy with this insurance.",
    "This service is slow and frustrating.",
    "The support team was helpful."
]

results = classifier(texts)

# DataFrame으로 정리
df = pd.DataFrame([
    {
        "문장": text,
        "감정": result["label"],
        "확률": round(result["score"], 4)
    }
    for text, result in zip(texts, results)
])

print(df.to_string(index=False))

                                   문장       감정     확률
 I am very happy with this insurance. POSITIVE 0.9999
This service is slow and frustrating. NEGATIVE 0.9996
        The support team was helpful. POSITIVE 0.9996
